<a href="https://colab.research.google.com/github/aiserhucui/news-recommendation-demo/blob/main/Video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Production-Ready Video Action Recognition Model
================================================
A comprehensive ML pipeline for video classification using PyTorch

Features:
- Data preparation from public videos
- Advanced preprocessing and augmentation
- 3D CNN architecture for spatiotemporal learning
- Comprehensive metrics and visualization
- Gradio interface for deployment
"""

# ============================================================================
# SECTION 1: SETUP AND INSTALLATIONS
# ============================================================================

# Install required packages
!pip install torch torchvision torchaudio --quiet
!pip install gradio opencv-python-headless pytorchvideo --quiet
!pip install tensorboard scikit-learn matplotlib seaborn --quiet
!pip install gdown yt-dlp --quiet

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, f1_score, precision_score,
                             recall_score, roc_curve, auc)
from sklearn.preprocessing import label_binarize
import gradio as gr
from tqdm.auto import tqdm
import warnings
import json
import pickle
from datetime import datetime
import logging

warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Set random seeds for reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")

# ============================================================================
# SECTION 2: DATA PREPARATION
# ============================================================================

class VideoDatasetPreparation:
    """Handles downloading and organizing video data"""

    def __init__(self, root_dir='./video_data'):
        self.root_dir = Path(root_dir)
        self.root_dir.mkdir(parents=True, exist_ok=True)

        # Define action classes for demonstration
        self.classes = [
            'jumping', 'running', 'walking', 'waving', 'sitting'
        ]

        # URLs for sample videos (using Pexels free stock videos)
        self.sample_videos = {
            'jumping': [
                'https://www.pexels.com/download/video/3044127/',
                'https://www.pexels.com/download/video/4761244/',
            ],
            'running': [
                'https://www.pexels.com/download/video/2795750/',
                'https://www.pexels.com/download/video/3764394/',
            ],
            'walking': [
                'https://www.pexels.com/download/video/3044148/',
                'https://www.pexels.com/download/video/3044154/',
            ],
            'waving': [
                'https://www.pexels.com/download/video/3044126/',
                'https://www.pexels.com/download/video/5524471/',
            ],
            'sitting': [
                'https://www.pexels.com/download/video/3044125/',
                'https://www.pexels.com/download/video/4761242/',
            ]
        }

    def create_synthetic_dataset(self, n_videos_per_class=20):
        """
        Create synthetic video dataset for demonstration
        In production, replace with actual video downloads
        """
        logger.info("Creating synthetic video dataset...")

        for class_name in self.classes:
            class_dir = self.root_dir / class_name
            class_dir.mkdir(exist_ok=True)

            for i in range(n_videos_per_class):
                video_path = class_dir / f"{class_name}_{i:03d}.mp4"

                if not video_path.exists():
                    self._generate_synthetic_video(video_path, class_name)

        logger.info(f"Dataset created at {self.root_dir}")
        return self.get_dataset_info()

    def _generate_synthetic_video(self, video_path, action_class):
        """Generate synthetic video for demonstration"""
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(str(video_path), fourcc, 30.0, (224, 224))

        num_frames = np.random.randint(60, 120)

        for frame_idx in range(num_frames):
            frame = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)

            # Add some pattern based on action class
            if action_class == 'jumping':
                y_pos = int(112 + 50 * np.sin(frame_idx * 0.2))
                cv2.circle(frame, (112, y_pos), 30, (255, 0, 0), -1)
            elif action_class == 'running':
                x_pos = int((frame_idx * 5) % 224)
                cv2.circle(frame, (x_pos, 112), 30, (0, 255, 0), -1)
            elif action_class == 'walking':
                x_pos = int((frame_idx * 2) % 224)
                cv2.circle(frame, (x_pos, 112), 30, (0, 0, 255), -1)
            elif action_class == 'waving':
                x_pos = int(112 + 30 * np.sin(frame_idx * 0.3))
                cv2.circle(frame, (x_pos, 112), 30, (255, 255, 0), -1)
            else:  # sitting
                cv2.circle(frame, (112, 150), 30, (255, 0, 255), -1)

            out.write(frame)

        out.release()

    def get_dataset_info(self):
        """Get information about the dataset"""
        info = {'total_videos': 0, 'classes': {}}

        for class_name in self.classes:
            class_dir = self.root_dir / class_name
            if class_dir.exists():
                videos = list(class_dir.glob('*.mp4'))
                info['classes'][class_name] = len(videos)
                info['total_videos'] += len(videos)

        return info

# ============================================================================
# SECTION 3: FEATURE ENGINEERING & PREPROCESSING
# ============================================================================

class VideoPreprocessor:
    """Advanced video preprocessing and augmentation"""

    def __init__(self, target_frames=16, target_size=(112, 112)):
        self.target_frames = target_frames
        self.target_size = target_size

        # Define augmentation pipelines
        self.train_transforms = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomResizedCrop(target_size[0], scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                 saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])

        self.val_transforms = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(target_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])

    def load_video(self, video_path):
        """Load video and extract frames"""
        cap = cv2.VideoCapture(str(video_path))
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        cap.release()
        return np.array(frames)

    def temporal_sampling(self, frames, method='uniform'):
        """Sample frames temporally"""
        total_frames = len(frames)

        if total_frames >= self.target_frames:
            if method == 'uniform':
                indices = np.linspace(0, total_frames - 1,
                                    self.target_frames, dtype=int)
            elif method == 'random':
                indices = np.sort(np.random.choice(total_frames,
                                                  self.target_frames,
                                                  replace=False))
        else:
            # Repeat frames if video is too short
            indices = np.arange(total_frames)
            while len(indices) < self.target_frames:
                indices = np.concatenate([indices, np.arange(total_frames)])
            indices = indices[:self.target_frames]

        return frames[indices]

    def extract_optical_flow(self, frames):
        """Extract optical flow features"""
        flows = []
        gray_prev = cv2.cvtColor(frames[0], cv2.COLOR_RGB2GRAY)

        for i in range(1, len(frames)):
            gray_curr = cv2.cvtColor(frames[i], cv2.COLOR_RGB2GRAY)
            flow = cv2.calcOpticalFlowFarneback(
                gray_prev, gray_curr, None,
                pyr_scale=0.5, levels=3, winsize=15,
                iterations=3, poly_n=5, poly_sigma=1.2, flags=0
            )
            flows.append(flow)
            gray_prev = gray_curr

        return np.array(flows)

    def preprocess(self, video_path, augment=False):
        """Complete preprocessing pipeline"""
        # Load video
        frames = self.load_video(video_path)

        # Temporal sampling
        frames = self.temporal_sampling(frames,
                                       method='random' if augment else 'uniform')

        # Apply transforms
        transform = self.train_transforms if augment else self.val_transforms
        processed_frames = []

        for frame in frames:
            processed_frame = transform(frame)
            processed_frames.append(processed_frame)

        # Stack frames: (T, C, H, W)
        video_tensor = torch.stack(processed_frames)

        return video_tensor


class VideoDataset(Dataset):
    """Custom Dataset for video classification"""

    def __init__(self, video_paths, labels, preprocessor, augment=False):
        self.video_paths = video_paths
        self.labels = labels
        self.preprocessor = preprocessor
        self.augment = augment

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        try:
            video_tensor = self.preprocessor.preprocess(video_path,
                                                       augment=self.augment)
            return video_tensor, label
        except Exception as e:
            logger.error(f"Error loading video {video_path}: {e}")
            # Return a blank video tensor on error
            return torch.zeros(self.preprocessor.target_frames, 3,
                             *self.preprocessor.target_size), label


# ============================================================================
# SECTION 4: DATA VISUALIZATION & ANALYSIS
# ============================================================================

class DataVisualizer:
    """Comprehensive data visualization"""

    @staticmethod
    def plot_dataset_distribution(dataset_info):
        """Plot class distribution"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        classes = list(dataset_info['classes'].keys())
        counts = list(dataset_info['classes'].values())

        # Bar plot
        ax1.bar(classes, counts, color='steelblue', alpha=0.8)
        ax1.set_xlabel('Action Class', fontsize=12)
        ax1.set_ylabel('Number of Videos', fontsize=12)
        ax1.set_title('Dataset Class Distribution', fontsize=14, fontweight='bold')
        ax1.tick_params(axis='x', rotation=45)

        # Pie chart
        colors = plt.cm.Set3(range(len(classes)))
        ax2.pie(counts, labels=classes, autopct='%1.1f%%',
               colors=colors, startangle=90)
        ax2.set_title('Class Proportion', fontsize=14, fontweight='bold')

        plt.tight_layout()
        plt.savefig('dataset_distribution.png', dpi=300, bbox_inches='tight')
        plt.show()

        logger.info(f"Total videos: {dataset_info['total_videos']}")

    @staticmethod
    def visualize_video_samples(video_paths, labels, class_names, n_samples=5):
        """Visualize sample frames from videos"""
        fig, axes = plt.subplots(n_samples, 4, figsize=(15, 3*n_samples))

        for i in range(min(n_samples, len(video_paths))):
            cap = cv2.VideoCapture(str(video_paths[i]))
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

            frame_indices = np.linspace(0, total_frames-1, 4, dtype=int)

            for j, frame_idx in enumerate(frame_indices):
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
                ret, frame = cap.read()

                if ret:
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    axes[i, j].imshow(frame_rgb)
                    axes[i, j].axis('off')

                    if j == 0:
                        axes[i, j].set_title(
                            f"{class_names[labels[i]]}\n Frame {frame_idx}",
                            fontsize=10
                        )
                    else:
                        axes[i, j].set_title(f"Frame {frame_idx}", fontsize=10)

            cap.release()

        plt.tight_layout()
        plt.savefig('video_samples.png', dpi=300, bbox_inches='tight')
        plt.show()

    @staticmethod
    def plot_frame_statistics(video_paths):
        """Analyze and plot video statistics"""
        durations = []
        frame_counts = []
        fps_list = []

        for video_path in tqdm(video_paths[:50], desc="Analyzing videos"):
            cap = cv2.VideoCapture(str(video_path))
            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            duration = frame_count / fps if fps > 0 else 0

            durations.append(duration)
            frame_counts.append(frame_count)
            fps_list.append(fps)
            cap.release()

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        axes[0].hist(durations, bins=20, color='skyblue', edgecolor='black')
        axes[0].set_xlabel('Duration (seconds)')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Video Duration Distribution')

        axes[1].hist(frame_counts, bins=20, color='lightcoral', edgecolor='black')
        axes[1].set_xlabel('Number of Frames')
        axes[1].set_ylabel('Frequency')
        axes[1].set_title('Frame Count Distribution')

        axes[2].hist(fps_list, bins=20, color='lightgreen', edgecolor='black')
        axes[2].set_xlabel('FPS')
        axes[2].set_ylabel('Frequency')
        axes[2].set_title('FPS Distribution')

        plt.tight_layout()
        plt.savefig('video_statistics.png', dpi=300, bbox_inches='tight')
        plt.show()

        logger.info(f"Avg duration: {np.mean(durations):.2f}s")
        logger.info(f"Avg frames: {np.mean(frame_counts):.0f}")
        logger.info(f"Avg FPS: {np.mean(fps_list):.1f}")


# ============================================================================
# SECTION 5: MODEL ARCHITECTURE
# ============================================================================

class C3D(nn.Module):
    """
    3D Convolutional Network for video classification
    Based on the C3D architecture
    """

    def __init__(self, num_classes=5, dropout=0.5):
        super(C3D, self).__init__()

        # Conv layer 1
        self.conv1 = nn.Conv3d(3, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))

        # Conv layer 2
        self.conv2 = nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool2 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        # Conv layer 3
        self.conv3a = nn.Conv3d(128, 256, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv3b = nn.Conv3d(256, 256, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool3 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        # Conv layer 4
        self.conv4a = nn.Conv3d(256, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv4b = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool4 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        # Conv layer 5
        self.conv5a = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv5b = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool5 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2),
                                  padding=(0, 1, 1))

        # Fully connected layers
        self.fc6 = nn.Linear(8192, 4096)
        self.fc7 = nn.Linear(4096, 4096)
        self.fc8 = nn.Linear(4096, num_classes)

        self.dropout = nn.Dropout(p=dropout)
        self.relu = nn.ReLU()

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out',
                                       nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # Input: (batch, time, channels, height, width)
        # Convert to (batch, channels, time, height, width) for Conv3d
        x = x.permute(0, 2, 1, 3, 4)

        # Convolutional layers
        x = self.relu(self.conv1(x))
        x = self.pool1(x)

        x = self.relu(self.conv2(x))
        x = self.pool2(x)

        x = self.relu(self.conv3a(x))
        x = self.relu(self.conv3b(x))
        x = self.pool3(x)

        x = self.relu(self.conv4a(x))
        x = self.relu(self.conv4b(x))
        x = self.pool4(x)

        x = self.relu(self.conv5a(x))
        x = self.relu(self.conv5b(x))
        x = self.pool5(x)

        # Flatten
        x = x.view(x.size(0), -1)

        # Fully connected layers
        x = self.relu(self.fc6(x))
        x = self.dropout(x)

        x = self.relu(self.fc7(x))
        x = self.dropout(x)

        x = self.fc8(x)

        return x


# ============================================================================
# SECTION 6: TRAINING PIPELINE
# ============================================================================

class MetricsTracker:
    """Track and compute training metrics"""

    def __init__(self):
        self.reset()

    def reset(self):
        self.losses = []
        self.accuracies = []
        self.precisions = []
        self.recalls = []
        self.f1_scores = []

    def update(self, loss, preds, labels):
        self.losses.append(loss)

        preds_np = preds.cpu().numpy()
        labels_np = labels.cpu().numpy()

        self.accuracies.append(accuracy_score(labels_np, preds_np))
        self.precisions.append(precision_score(labels_np, preds_np,
                                              average='weighted', zero_division=0))
        self.recalls.append(recall_score(labels_np, preds_np,
                                        average='weighted', zero_division=0))
        self.f1_scores.append(f1_score(labels_np, preds_np,
                                      average='weighted', zero_division=0))

    def get_average_metrics(self):
        return {
            'loss': np.mean(self.losses),
            'accuracy': np.mean(self.accuracies),
            'precision': np.mean(self.precisions),
            'recall': np.mean(self.recalls),
            'f1_score': np.mean(self.f1_scores)
        }


class VideoTrainer:
    """Production-ready training pipeline"""

    def __init__(self, model, train_loader, val_loader, class_names,
                 lr=0.001, device='cuda'):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.class_names = class_names
        self.device = device

        # Loss and optimizer
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr,
                                         weight_decay=1e-5)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=3
        )

        # Metrics tracking
        self.train_history = {'loss': [], 'accuracy': [], 'f1_score': []}
        self.val_history = {'loss': [], 'accuracy': [], 'f1_score': []}
        self.best_val_acc = 0.0

        logger.info(f"Trainer initialized. Model has {self._count_parameters():,} parameters")

    def _count_parameters(self):
        return sum(p.numel() for p in self.model.parameters() if p.requires_grad)

    def train_epoch(self, epoch):
        """Train for one epoch"""
        self.model.train()
        metrics = MetricsTracker()

        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch} [Train]')
        for videos, labels in pbar:
            videos, labels = videos.to(self.device), labels.to(self.device)

            # Forward pass
            self.optimizer.zero_grad()
            outputs = self.model(videos)
            loss = self.criterion(outputs, labels)

            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()

            # Calculate metrics
            preds = outputs.argmax(dim=1)
            metrics.update(loss.item(), preds, labels)

            # Update progress bar
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        return metrics.get_average_metrics()

    def validate(self, epoch):
        """Validate the model"""
        self.model.eval()
        metrics = MetricsTracker()

        all_preds = []
        all_labels = []

        with torch.no_grad():
            pbar = tqdm(self.val_loader, desc=f'Epoch {epoch} [Val]')
            for videos, labels in pbar:
                videos, labels = videos.to(self.device), labels.to(self.device)

                outputs = self.model(videos)
                loss = self.criterion(outputs, labels)

                preds = outputs.argmax(dim=1)
                metrics.update(loss.item(), preds, labels)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

                pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        avg_metrics = metrics.get_average_metrics()
        return avg_metrics, np.array(all_preds), np.array(all_labels)

    def train(self, num_epochs=20, save_best=True):
        """Complete training loop"""
        logger.info(f"Starting training for {num_epochs} epochs...")

        for epoch in range(1, num_epochs + 1):
            # Train
            train_metrics = self.train_epoch(epoch)

            # Validate
            val_metrics, val_preds, val_labels = self.validate(epoch)

            # Learning rate scheduling
            self.scheduler.step(val_metrics['loss'])

            # Save metrics
            self.train_history['loss'].append(train_metrics['loss'])
            self.train_history['accuracy'].append(train_metrics['accuracy'])
            self.train_history['f1_score'].append(train_metrics['f1_score'])

            self.val_history['loss'].append(val_metrics['loss'])
            self.val_history['accuracy'].append(val_metrics['accuracy'])
            self.val_history['f1_score'].append(val_metrics['f1_score'])

            # Log results
            logger.info(f"\nEpoch {epoch}/{num_epochs}")
            logger.info(f"Train - Loss: {train_metrics['loss']:.4f}, "
                       f"Acc: {train_metrics['accuracy']:.4f}, "
                       f"F1: {train_metrics['f1_score']:.4f}")
            logger.info(f"Val   - Loss: {val_metrics['loss']:.4f}, "
                       f"Acc: {val_metrics['accuracy']:.4f}, "
                       f"F1: {val_metrics['f1_score']:.4f}")

            # Save best model
            if save_best and val_metrics['accuracy'] > self.best_val_acc:
                self.best_val_acc = val_metrics['accuracy']
                self.save_checkpoint('best_model.pth', epoch, val_metrics)
                logger.info(f"✓ Best model saved! Val Acc: {self.best_val_acc:.4f}")

        logger.info("\nTraining completed!")
        return self.train_history, self.val_history

    def save_checkpoint(self, filename, epoch, metrics):
        """Save model checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'metrics': metrics,
            'class_names': self.class_names
        }
        torch.save(checkpoint, filename)

    def load_checkpoint(self, filename):
        """Load model checkpoint"""
        checkpoint = torch.load(filename, map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        return checkpoint


# ============================================================================
# SECTION 7: EVALUATION & VISUALIZATION
# ============================================================================

class ModelEvaluator:
    """Comprehensive model evaluation"""

    @staticmethod
    def plot_training_history(train_history, val_history):
        """Plot training curves"""
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        epochs = range(1, len(train_history['loss']) + 1)

        metrics = [('loss', 'Loss'), ('accuracy', 'Accuracy'),
                  ('f1_score', 'F1 Score')]

        for idx, (metric, title) in enumerate(metrics):
            axes[idx].plot(epochs, train_history[metric], 'b-',
                          label='Train', linewidth=2)
            axes[idx].plot(epochs, val_history[metric], 'r-',
                          label='Validation', linewidth=2)
            axes[idx].set_xlabel('Epoch', fontsize=12)
            axes[idx].set_ylabel(title, fontsize=12)
            axes[idx].set_title(f'{title} Over Epochs',
                              fontsize=14, fontweight='bold')
            axes[idx].legend()
            axes[idx].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
        plt.show()

    @staticmethod
    def plot_confusion_matrix(y_true, y_pred, class_names):
        """Plot confusion matrix"""
        cm = confusion_matrix(y_true, y_pred)

        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=class_names, yticklabels=class_names,
                   cbar_kws={'label': 'Count'})
        plt.xlabel('Predicted Label', fontsize=12)
        plt.ylabel('True Label', fontsize=12)
        plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
        plt.show()

        # Calculate per-class accuracy
        per_class_acc = cm.diagonal() / cm.sum(axis=1)
        for i, acc in enumerate(per_class_acc):
            logger.info(f"{class_names[i]}: {acc:.2%} accuracy")

    @staticmethod
    def plot_classification_report(y_true, y_pred, class_names):
        """Visualize classification report"""
        report = classification_report(y_true, y_pred,
                                      target_names=class_names,
                                      output_dict=True)

        metrics = ['precision', 'recall', 'f1-score']
        data = []

        for class_name in class_names:
            data.append([report[class_name][m] for m in metrics])

        data = np.array(data)

        fig, ax = plt.subplots(figsize=(10, 6))
        x = np.arange(len(class_names))
        width = 0.25

        for i, metric in enumerate(metrics):
            ax.bar(x + i*width, data[:, i], width, label=metric.capitalize())

        ax.set_xlabel('Class', fontsize=12)
        ax.set_ylabel('Score', fontsize=12)
        ax.set_title('Classification Report', fontsize=14, fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(class_names, rotation=45)
        ax.legend()
        ax.grid(axis='y', alpha=0.3)

        plt.tight_layout()
        plt.savefig('classification_report.png', dpi=300, bbox_inches='tight')
        plt.show()

    @staticmethod
    def evaluate_model(model, data_loader, class_names, device):
        """Complete model evaluation"""
        model.eval()
        all_preds = []
        all_labels = []
        all_probs = []

        with torch.no_grad():
            for videos, labels in tqdm(data_loader, desc='Evaluating'):
                videos = videos.to(device)
                outputs = model(videos)
                probs = F.softmax(outputs, dim=1)
                preds = outputs.argmax(dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.numpy())
                all_probs.extend(probs.cpu().numpy())

        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        all_probs = np.array(all_probs)

        # Calculate metrics
        accuracy = accuracy_score(all_labels, all_preds)

        logger.info(f"\n{'='*60}")
        logger.info("FINAL EVALUATION RESULTS")
        logger.info(f"{'='*60}")
        logger.info(f"Overall Accuracy: {accuracy:.4f}")
        logger.info(f"\n{classification_report(all_labels, all_preds, target_names=class_names)}")

        return all_preds, all_labels, all_probs


# ============================================================================
# SECTION 8: GRADIO INTERFACE
# ============================================================================

class VideoClassifierApp:
    """Gradio interface for video classification"""

    def __init__(self, model, preprocessor, class_names, device):
        self.model = model
        self.preprocessor = preprocessor
        self.class_names = class_names
        self.device = device
        self.model.eval()

    def predict(self, video_path):
        """Predict action from video"""
        try:
            # Preprocess video
            video_tensor = self.preprocessor.preprocess(video_path, augment=False)
            video_tensor = video_tensor.unsqueeze(0).to(self.device)

            # Predict
            with torch.no_grad():
                outputs = self.model(video_tensor)
                probs = F.softmax(outputs, dim=1)
                pred_idx = outputs.argmax(dim=1).item()

            # Format results
            confidence_scores = {
                self.class_names[i]: float(probs[0][i])
                for i in range(len(self.class_names))
            }

            predicted_class = self.class_names[pred_idx]
            confidence = float(probs[0][pred_idx])

            # Create result text
            result_text = f"**Predicted Action:** {predicted_class}\n"
            result_text += f"**Confidence:** {confidence:.2%}\n\n"
            result_text += "**All Probabilities:**\n"
            for class_name, prob in sorted(confidence_scores.items(),
                                          key=lambda x: x[1], reverse=True):
                result_text += f"- {class_name}: {prob:.2%}\n"

            return result_text, confidence_scores

        except Exception as e:
            return f"Error processing video: {str(e)}", {}

    def launch(self):
        """Launch Gradio interface"""
        with gr.Blocks(theme=gr.themes.Soft()) as demo:
            gr.Markdown("""
            # 🎬 Video Action Recognition System
            ### Production-Ready ML Model for Video Classification
            Upload a video to classify the action being performed.
            """)

            with gr.Row():
                with gr.Column():
                    video_input = gr.Video(label="Upload Video")
                    predict_btn = gr.Button("Classify Action", variant="primary")

                with gr.Column():
                    output_text = gr.Markdown(label="Prediction Results")
                    output_plot = gr.Label(label="Confidence Scores", num_top_classes=5)

            gr.Markdown("""
            ### Supported Actions:
            - 🏃 Running
            - 🚶 Walking
            - 🦘 Jumping
            - 👋 Waving
            - 🪑 Sitting

            ### Model Details:
            - **Architecture:** C3D (3D Convolutional Network)
            - **Input:** 16 frames at 112x112 resolution
            - **Features:** Spatiotemporal learning with 3D convolutions
            """)

            predict_btn.click(
                fn=self.predict,
                inputs=[video_input],
                outputs=[output_text, output_plot]
            )

        return demo


# ============================================================================
# SECTION 9: MAIN EXECUTION PIPELINE
# ============================================================================

def main():
    """Main execution pipeline"""

    logger.info("="*70)
    logger.info("VIDEO ACTION RECOGNITION - PRODUCTION ML PIPELINE")
    logger.info("="*70)

    # ========== STEP 1: DATA PREPARATION ==========
    logger.info("\n[STEP 1/8] Preparing dataset...")
    data_prep = VideoDatasetPreparation(root_dir='./video_data')
    dataset_info = data_prep.create_synthetic_dataset(n_videos_per_class=20)

    # ========== STEP 2: DATA VISUALIZATION ==========
    logger.info("\n[STEP 2/8] Visualizing dataset...")
    visualizer = DataVisualizer()
    visualizer.plot_dataset_distribution(dataset_info)

    # Collect all video paths and labels
    video_paths = []
    labels = []
    class_names = data_prep.classes

    for class_idx, class_name in enumerate(class_names):
        class_dir = data_prep.root_dir / class_name
        class_videos = list(class_dir.glob('*.mp4'))
        video_paths.extend(class_videos)
        labels.extend([class_idx] * len(class_videos))

    # Visualize samples
    visualizer.visualize_video_samples(video_paths[:5], labels[:5], class_names)
    visualizer.plot_frame_statistics(video_paths)

    # ========== STEP 3: TRAIN/VAL SPLIT ==========
    logger.info("\n[STEP 3/8] Splitting dataset...")
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        video_paths, labels, test_size=0.2, random_state=42, stratify=labels
    )

    logger.info(f"Training samples: {len(train_paths)}")
    logger.info(f"Validation samples: {len(val_paths)}")

    # ========== STEP 4: DATA PREPROCESSING ==========
    logger.info("\n[STEP 4/8] Setting up data preprocessing...")
    preprocessor = VideoPreprocessor(target_frames=16, target_size=(112, 112))

    # Create datasets
    train_dataset = VideoDataset(train_paths, train_labels,
                                 preprocessor, augment=True)
    val_dataset = VideoDataset(val_paths, val_labels,
                               preprocessor, augment=False)

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=4,
                             shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=4,
                           shuffle=False, num_workers=0)

    # ========== STEP 5: MODEL INITIALIZATION ==========
    logger.info("\n[STEP 5/8] Initializing model...")
    model = C3D(num_classes=len(class_names), dropout=0.5)

    # ========== STEP 6: MODEL TRAINING ==========
    logger.info("\n[STEP 6/8] Training model...")
    trainer = VideoTrainer(model, train_loader, val_loader,
                          class_names, lr=0.001, device=device)

    # Train model (reduced epochs for demonstration)
    train_history, val_history = trainer.train(num_epochs=10, save_best=True)

    # ========== STEP 7: MODEL EVALUATION ==========
    logger.info("\n[STEP 7/8] Evaluating model...")
    evaluator = ModelEvaluator()

    # Plot training history
    evaluator.plot_training_history(train_history, val_history)

    # Load best model
    checkpoint = trainer.load_checkpoint('best_model.pth')

    # Comprehensive evaluation
    val_preds, val_labels, val_probs = evaluator.evaluate_model(
        model, val_loader, class_names, device
    )

    # Plot evaluation metrics
    evaluator.plot_confusion_matrix(val_labels, val_preds, class_names)
    evaluator.plot_classification_report(val_labels, val_preds, class_names)

    # ========== STEP 8: GRADIO INTERFACE ==========
    logger.info("\n[STEP 8/8] Launching Gradio interface...")
    app = VideoClassifierApp(model, preprocessor, class_names, device)
    demo = app.launch()

    logger.info("\n" + "="*70)
    logger.info("✓ Pipeline completed successfully!")
    logger.info("="*70)

    # Launch the interface
    demo.launch(share=True, debug=True)

    return {
        'model': model,
        'preprocessor': preprocessor,
        'class_names': class_names,
        'train_history': train_history,
        'val_history': val_history,
        'demo': demo
    }


# Run the complete pipeline
if __name__ == "__main__":
    results = main()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━ 15.7/36.3 MB 246.1 MB/s eta 0:00:01